# B04 · S1 — El robot y su cinemática (Célula-07)

**Objetivo (RA4-a):** entender qué es un robot, su hardware y su **cinemática** (FK, IK, jacobiano, singularidades), ejecutando el código en Colab.

> Práctica guiada de la Sesión 1 de los [apuntes](../apuntes.md). Hilo conductor: **Célula-07** (brazo de *pick-and-place* + AMR).

## 1. Brazo plano 3R: cinemática directa (FK)

De los ángulos a la posición del efector.

In [ ]:
import numpy as np
from scipy.optimize import least_squares

def fk_3r(q, L=(1.0, 1.0, 1.0)):
    t1, t2, t3 = q
    x = L[0]*np.cos(t1) + L[1]*np.cos(t1+t2) + L[2]*np.cos(t1+t2+t3)
    y = L[0]*np.sin(t1) + L[1]*np.sin(t1+t2) + L[2]*np.sin(t1+t2+t3)
    return np.array([x, y])

print("FK q=0:      ", np.round(fk_3r([0, 0, 0]), 3))
print("FK q=90grados:", np.round(fk_3r([np.pi/2, 0, 0]), 3))

## 2. Jacobiano y singularidades

El jacobiano relaciona velocidades articulares y cartesianas (`v = J·q̇`). Cuando pierde rango (codo estirado), la velocidad articular se dispara: **singularidad**.

In [ ]:
def jacobian_2d(q, L=(1.0, 1.0, 1.0)):
    t1, t2, t3 = q
    J = np.zeros((2, 3))
    J[0, 0] = -L[0]*np.sin(t1) - L[1]*np.sin(t1+t2) - L[2]*np.sin(t1+t2+t3)
    J[0, 1] = -L[1]*np.sin(t1+t2) - L[2]*np.sin(t1+t2+t3)
    J[0, 2] = -L[2]*np.sin(t1+t2+t3)
    J[1, 0] =  L[0]*np.cos(t1) + L[1]*np.cos(t1+t2) + L[2]*np.cos(t1+t2+t3)
    J[1, 1] =  L[1]*np.cos(t1+t2) + L[2]*np.cos(t1+t2+t3)
    J[1, 2] =  L[2]*np.cos(t1+t2+t3)
    return J

for q in ([0, 0, 0], [0.3, 0.4, 0.5], [0, 0, 0.01]):
    J = jacobian_2d(q)
    print("q=", q, "| det(JJ^T) =", round(float(np.linalg.det(J @ J.T)), 4))
# det(JJ^T) ~ 0 => configuracion singular (codo estirado / alineado)

## 3. Cinemática inversa (IK) numérica

De la posición deseada a los ángulos. Puede haber **múltiples soluciones**: cambia el punto de partida y observa.

In [ ]:
def ik_3r(target, q0=(0.1, 0.1, 0.1)):
    sol = least_squares(lambda q: fk_3r(q) - target, q0)
    return sol.x

for q0 in [(0.1, 0.1, 0.1), (1.0, -1.0, 0.5)]:
    q = ik_3r([2.0, 1.0], q0)
    print("q0=", q0, "-> q=", np.round(q, 3), "| FK=", np.round(fk_3r(q), 3))

## 4. Un robot real con `roboticstoolbox-python`

El modelo **Panda** (Franka Emika) trae su cinemática resuelta.

In [ ]:
%pip install roboticstoolbox-python spatialmath-python
import roboticstoolbox as rtb

robot = rtb.models.Panda()
print(robot)

pose = robot.fkine([0, -0.8, 0.8, 0, 0.8, 0, 0])   # FK
print("Pose:\n", pose)

sol = robot.ikine_LM(pose)                          # IK (Levenberg-Marquardt)
print("IK ok:", sol.success, "| q =", [round(x, 2) for x in sol.q])

## Actividad guiada

1. Cambia `q` en la FK del brazo 3R y dibuja la posición (con `matplotlib`).
2. Encuentra por búsqueda una configuración de **singularidad** (donde `det(JJ^T) ≈ 0`).
3. Con el Panda, resuelve la IK para una pose nueva y comprueba con `fkine`.

**Alcance de la sesión:** tipos y hardware, jerarquía tarea→movimiento→control, FK/IK, singularidades y repetibilidad.